# 1.1 — Strengths and failures
**Session 1.1 · Demo**

What you'll see: six prompts sent to Claude. The first two show what LLMs do well.
The next three show how they fail. The last one shows a simple fix.

In [ ]:
# Setup: install the Anthropic SDK and connect with your API key.
import importlib.util, shutil, subprocess, sys

if importlib.util.find_spec("anthropic") is None:
    installer = (
        ["uv", "pip", "install", "--python", sys.executable]
        if shutil.which("uv")
        else [sys.executable, "-m", "pip", "install"]
    )
    subprocess.check_call([*installer, "-q", "anthropic"])

import os
from getpass import getpass

import anthropic


def read_api_key():
    """Environment variable, then the Colab secret, then a prompt. The key is never printed."""
    key = os.environ.get("ANTHROPIC_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata

        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        return getpass("Paste your Anthropic API key: ")


client = anthropic.Anthropic(api_key=read_api_key())
MODEL = "claude-sonnet-5-5"
print(f"Ready. Model: {MODEL}")

Ready. Model: claude-sonnet-5-5


## Talking to Claude
Every prompt in this demo goes through `ask()`. One request to Claude needs three things:
the **model**, a **max_tokens** limit for the answer, and the **messages** (here, one user
message).
Answers vary a little from run to run.

By default Claude **thinks** before it answers. Here we turn thinking off, so we see the
model's first, instant answer: that's where the classic failures show up.

The reply comes back as a list of **content blocks**; we keep only the `text` blocks.
It also has its **usage**: how many tokens went in and came out. Tokens are what you pay for.

In [ ]:
import time

from IPython.display import Markdown, display


def ask(prompt):
    """Send one prompt to Claude, show the reply, and print the tokens and time it took."""
    request = {
        "model": MODEL,
        "max_tokens": 1000,
        "thinking": {"type": "between_tools"},  # Thinking off
        "messages": [{"role": "user", "content": prompt}],
    }

    start = time.perf_counter()
    response = client.messages.create(**request)
    seconds = time.perf_counter() - start

    answer = "".join(block.text for block in response.content if block.type == "text")
    display(Markdown(answer))
    print(
        f"{response.usage.input_tokens} tokens in · {response.usage.output_tokens} tokens out · {seconds:.1f}s"
    )
    return answer

## 1. Summarize a long policy
**Watch for:** a long document compressed into three bullets, in a tone we asked for.

In [ ]:
LONG_POLICY = """# Data Classification Policy

Everything Northgate Logistics stores, sends or prints belongs to one of four classes. The class decides where the information can live, who can see it, and how it can be shared. This policy applies to every employee, contractor and temporary worker, in all three countries where we operate, on any device. If you create a document, you are responsible for labelling it. If you receive one without a label, treat it as Internal until the owner tells you otherwise.

## The four classes

**Public.** Information that has been approved for release and would do no harm if anyone read it. Examples are published price lists, press releases, job adverts and the public website. Public data may be shared freely, but only the Communications team may approve something as Public.

**Internal.** The default for everyday business information that is not meant for the outside world. Examples are the staff directory, meeting notes, general procedures and most of the documents in this knowledge base. Internal data may be shared with anyone at Northgate, and with partners who have signed a non-disclosure agreement if there is a clear business need.

**Confidential.** Information that could damage the company, a customer or an employee if it leaked. Examples are customer contracts, shipment rates, non-public financial results, supplier negotiations and most personal data. Share it only with people who need it for their work. It must be encrypted when sent outside the company, and it may only be stored on shared drives, in the approved document system, or on an encrypted company laptop.

**Restricted.** The most sensitive information we hold, where a leak could cause serious legal, financial or safety harm. Examples are payroll files, salary bands, health records, identity documents, payment card data, encryption keys and the details of open security investigations. Access is limited to named people and is reviewed every quarter. Restricted data must never be sent by email or chat, must never be stored on a personal device, and must be kept in a dedicated encrypted folder created by IT Security.

## Handling rules at a glance

Printing: Confidential and Restricted documents must go through FollowMe printing and be collected immediately. Disposal: use the locked shredding bins for paper, and ask the help desk to wipe any storage device. Do not throw anything classified above Public into an ordinary bin.

AI tools: only approved tools may be used. Do not paste Confidential or Restricted data into a public chatbot, translation site or similar online service, even for a quick task. Check the Approved Software List first.

Travel: take only the data you need. Restricted data must not leave the country it is stored in without written approval from IT Security.

## If something goes wrong

If you send classified data to the wrong person, lose a device, or see data somewhere it should not be, report it to the help desk immediately, on the same day. Do not try to fix it quietly. Privacy law gives the company as little as 72 hours to notify regulators of some incidents, so an early report matters a great deal. Honest mistakes reported quickly are treated as learning opportunities, and hiding one is treated far more seriously.

## Questions and exceptions

Your manager should be your first stop when you are unsure how to label something. Exceptions to this policy need written approval from the IT Security team and expire after 12 months. The policy itself is reviewed once a year."""

print(f"The policy is {len(LONG_POLICY.split())} words.")

prompt = (
    "Summarize this policy for a new warehouse employee in three short, friendly bullet points."
    f"\n\n<policy>\n{LONG_POLICY}\n</policy>"
)
ask(prompt);

The policy is 580 words.


- **Label it and handle it by class:** Everything we store, send or print is Public, Internal, Confidential or Restricted. If you create a document, label it; if you get one with no label, treat it as Internal until the owner says otherwise. Sensitive stuff (like customer contracts, pay files or ID documents) gets extra care, so only share it with people who need it, never paste it into public chatbots or online tools, and use FollowMe printing and the locked shredding bins instead of regular printers and bins.
- **Not sure? Ask your manager first:** If you're unsure how to label or share something, your manager is your go-to before you guess.
- **Made a mistake? Tell the help desk the same day:** If you send something to the wrong person, lose a device, or spot data where it shouldn't be, report it right away. Honest mistakes reported quickly are treated as learning moments, but hiding one is treated much more seriously.

1072 tokens in · 295 tokens out · 2.8s


## 2. Extract fields into JSON
**Watch for:** clean structure from a messy email. Check how it handles the date and the
phone extension.

Models sometimes wrap JSON in a code fence or add a sentence around it, so `parse_json()`
keeps only the part between the first `{` and the last `}` before reading it.

In [ ]:
import json

MESSY_EMAIL = """hi helpdesk!! sorry for the mess, on my phone between loads. its Tomas from the
Hamburg depot (t.brenner@northgate.example). laptop NG-48213 wont charge since monday,
tried 2 chargers. i need it fixed before the audit on 24 nov or im in trouble, boss says
its urgent. can use a loaner till then. oh and ext is 5512 if u need to call. thx"""

prompt = (
    "Extract these fields from the email and return only a JSON object: "
    "name, email, office, asset_tag, problem, needed_by, phone_extension, priority, accepts_loaner."
    "the needed_by field should be a date"
    f"\n\n<email>\n{MESSY_EMAIL}\n</email>"
)
answer = ask(prompt)


def parse_json(text):
    """Read the JSON object inside a reply, ignoring anything around it."""
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start : end + 1])


print(json.dumps(parse_json(answer), indent=2))

```json
{
  "name": "Tomas Brenner",
  "email": "t.brenner@northgate.example",
  "office": "Hamburg depot",
  "asset_tag": "NG-48213",
  "problem": "Laptop won't charge since Monday; tried 2 chargers",
  "needed_by": "2025-11-24",
  "phone_extension": "5512",
  "priority": "urgent",
  "accepts_loaner": true
}
```

219 tokens in · 154 tokens out · 1.5s
{
  "name": "Tomas Brenner",
  "email": "t.brenner@northgate.example",
  "office": "Hamburg depot",
  "asset_tag": "NG-48213",
  "problem": "Laptop won't charge since Monday; tried 2 chargers",
  "needed_by": "2025-11-24",
  "phone_extension": "5512",
  "priority": "urgent",
  "accepts_loaner": true
}


## 3. Count letters
**Watch for:** confident numbers that are sometimes wrong, and change from one try to the next.

Here we use the small, fast model (Haiku) and ask for an **instant** answer: thinking off,
the number only, and no room to work it out. We ask each question three times.

In [ ]:
import re

SMALL_MODEL = "claude-haiku-5-5"


def quick_count(text, letter):
    """Ask the small model for an instant answer: thinking off, number only."""
    response = client.messages.create(
        model=SMALL_MODEL,
        max_tokens=10,
        thinking={"type": "disabled"},
        system="Reply with only the number. No working, no words.",
        messages=[
            {
                "role": "user",
                "content": f"How many times does the letter '{letter}' appear in this text?\n\n{text}",
            }
        ],
    )
    answer = "".join(block.text for block in response.content if block.type == "text")
    numbers = re.findall(r"\d+", answer)
    return int(numbers[0]) if numbers else None


TESTS = [
    ("strawberry", "r"),
    ("flibberquantumorphicastrazzle", "r"),  # Invented, so it can't be memorised.
    ("pneumonoultramicroscopicsilicovolcanoconiosis", "o"),  # Very long.
]

for text, letter in TESTS:
    answers = [quick_count(text, letter) for _ in range(3)]
    true_count = text.count(letter)
    verdict = (
        "right ✅" if all(answer == true_count for answer in answers) else "wrong ❌"
    )
    print(
        f"'{letter}' in {text}\n   Claude says {answers}, Python counts {true_count}: {verdict}\n"
    )

'r' in strawberry
   Claude says [3, 3, 3], Python counts 3: right ✅

'r' in flibberquantumorphicastrazzle
   Claude says [3, 3, 3], Python counts 3: right ✅

'o' in pneumonoultramicroscopicsilicovolcanoconiosis
   Claude says [None, None, None], Python counts 9: wrong ❌



**Why this happens:** Claude never sees letters. Before the model reads your text, the text is
cut into **tokens**: chunks of a few characters, such as `flib` + `ber` + `quant`. Each chunk
reaches the model as one unit (a number), not as a row of letters.

So to count the r's, the model has to *remember* how each chunk is spelled. It's like counting
the letters of words you have only ever heard, never seen written. More chunks, more chances
to slip.

Bigger models, or models allowed to think, usually get it right: they first spell the word out
(`f-l-i-b-b-e-r...`), so each letter becomes its own token, and then count. It works, but it
costs tokens and time. Python does it instantly, for free.

## 4. Ask about last week's news
**Watch for:** the knowledge cutoff. It may say it can't know, or it may offer older events.
Either way, it has no record of last week.

In [ ]:
from datetime import date

prompt = (
    f"Today is {date.today():%d %B %Y}. "
    "What were the three biggest technology news stories of the past week?"
)
ask(prompt);

I can't tell you what happened this past week. I don't have access to live news or the internet, and my training data has a cutoff that falls well before November 2026. Anything I listed as "the biggest stories of the week" would be invented, and with news that would be worse than no answer.

For reliable coverage of the past week, these are good places to look:

- **Tech news sites:** The Verge, Ars Technica, TechCrunch, Wired, and Reuters' technology section
- **Newsletters:** Platformer, Stratechery, or the daily briefings from The Verge or Axios
- **Aggregators:** Hacker News (news.ycombinator.com) shows what the tech community is discussing in real time, and Techmeme collects the day's top stories
- **A news-enabled assistant:** If you have access to an AI tool with web search, it can pull and summarize recent headlines.

If you paste in a few headlines or articles you've come across, I can help you summarize them, explain the background, or work out why they matter. I can also go over how a particular technology or company has developed up to my knowledge cutoff, if you're looking for context on something specific.

34 tokens in · 362 tokens out · 4.2s


## 5. Ask for a citation
**Watch for:** we ask for a study that doesn't exist. Current models usually refuse and say
why. They are trained to.

In [ ]:
CITATION_QUESTION = (
    "Which peer-reviewed study measured how forced 90-day password rotation changed "
    "help-desk ticket volume at mid-sized European logistics companies? "
    "Reply with the full citation in APA format, including the DOI, then the headline statistic."
)
ask(CITATION_QUESTION);

I can't point to a peer-reviewed study that matches that description, and I'd rather say so than give you a citation I might be fabricating. A paper specifically measuring help-desk ticket volume after forced 90-day password rotation at mid-sized European logistics companies isn't something I can identify. If I wrote out an APA citation with a DOI and a headline statistic, I'd essentially be inventing details that look authoritative but wouldn't hold up when you tried to look them up.

What I can tell you is what the surrounding literature looks like, since parts of your question connect to real research:

On the effects of forced password rotation generally, the most cited work is Zhang, Monrose, and Reiter (2010), "The Security of Modern Password Expiration: An Algorithmic Framework and Empirical Analysis," presented at ACM CCS. It showed that users who change passwords on a schedule tend to make predictable modifications, which weakens the security benefit. Chiasson and Biddle (2007) and later usability studies also document user burden from expiration policies.

On the policy shift, NIST Special Publication 800-63B (2017, updated since) moved away from mandatory periodic rotation, partly citing the usability and support costs. This is guidance, not a peer-reviewed study, but it summarizes the reasoning.

On help-desk costs, many figures about password-reset ticket volume come from industry sources like Forrester or Gartner, which are widely quoted but are analyst reports rather than peer-reviewed studies, and I wouldn't want to hand you a specific percentage without being sure of its provenance.

If you need something sector- and region-specific, the practical route is searching Google Scholar, IEEE Xplore, or ACM Digital Library with terms like "password expiration," "help desk," "support cost," and "logistics" or "SME." It's quite possible the exact study you have in mind exists but isn't one I can verify.

If you tell me where you came across this claim, I can help you work out whether it's likely to be real, or point you toward the closest verifiable sources.

80 tokens in · 666 tokens out · 6.3s


### The harder case: a real paper
Refusing is easy when a question looks suspicious. The risk is when the model *thinks* it knows.
We ask the model for an instant answer: the **DOI** of a real paper. A DOI is a paper's
permanent ID, and `https://doi.org/<DOI>` opens it.

**Watch for:** a DOI that looks perfect. Then we check which paper it really belongs to.

In [ ]:
PAPER = (
    'Zhang, Monrose & Reiter (2010), "The security of modern password expiration: '
    'an algorithmic framework and empirical analysis", ACM CCS'
)
REAL_DOI = "10.1145/1866307.1866328"

response = client.messages.create(
    model=SMALL_MODEL,
    max_tokens=60,
    thinking={"type": "disabled"},
    system="Reply with only the DOI.",
    messages=[{"role": "user", "content": f"What is the DOI of {PAPER}?"}],
)
answer = "".join(block.text for block in response.content if block.type == "text")
found = re.search(r"10\.\d{4,9}/[^\s,;)]+", answer)  # A DOI always starts with "10."
doi = found.group(0).rstrip(".") if found else None

print(f"Claude says:  {doi or answer}")
print(f"The real DOI: {REAL_DOI}")
print(f"Check it yourself: https://doi.org/{doi}")

Claude says:  10.1145/1866307.1866414
The real DOI: 10.1145/1866307.1866328
Check it yourself: https://doi.org/10.1145/1866307.1866414


## 6. Ask it to say "I don't know"
**Watch for:** we give Claude a document and a rule: answer only from the document, otherwise
say "I don't know".

In [ ]:
PASSWORD_POLICY = """# Password Reset Policy

This policy applies to all Northgate accounts. It replaces all earlier password guidance.

## Password rules

- Minimum length is 14 characters. We recommend a passphrase of several unrelated words.
- There are no required mixes of symbols or numbers.
- You cannot reuse any of your last 10 passwords.
- Never share your password, including with the help desk. We will never ask for it.

## How often you must change it

Passwords do not expire on a schedule. You do not need to change yours unless:

- you suspect someone else knows it
- IT Security asks you to, for example after a breach notice
- it appears on a list of compromised passwords, in which case you will be prompted at sign-in

## How to reset your password

Use the self-service page at password.northgate.example:

1. Enter your username.
2. Approve the request in the NorthKey Authenticator app.
3. Choose a new password that meets the rules above.

The new password takes effect within five minutes. Close and reopen Outlook and Teams if they keep asking you to sign in.

## If self-service does not work

If you lost your phone or cannot complete MFA, phone the help desk on extension 4357. The engineer will verify your identity using your employee number and your manager's name, then issue a temporary password valid for 24 hours.

If repeated failed attempts have locked your account, see the Account Lockout guide.

## If you think your password is compromised

Reset it straight away and report the event to the help desk as a security incident. If you typed your password into a suspicious page, phone us rather than opening a ticket, and see the Phishing Reporting guide."""

QUESTION = "After how many failed sign-in attempts does my account get locked, and for how long?"

query =f"""Answer only from the document provided. If the document does not contain
the answer, say "I don\'t know" and do not guess.

<document>\n{PASSWORD_POLICY}\n</document>

\n\n{QUESTION}
"""
ask(query);

I don't know. The document mentions that repeated failed attempts can lock an account, but it doesn't say how many attempts trigger a lockout or how long the lock lasts. It points to the Account Lockout guide for that topic.

600 tokens in · 73 tokens out · 1.0s
